# IEEE-CIS Fraud Detection — EDA

**Goal:** Understand the dataset before modelling.
- Class imbalance  
- Missing value rates  
- Transaction amount distribution  
- Feature correlations  

**Dataset:** `data/raw/train_transaction.csv` (590K rows from Kaggle)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_PATH = Path('../data/raw/train_transaction.csv')
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Shape: {df.shape}')
df.head(3)

## 1. Class Imbalance

In [ ]:
fraud_counts = df['isFraud'].value_counts()
fraud_pct = df['isFraud'].mean() * 100
print(f'Fraud rate: {fraud_pct:.2f}%')
print(fraud_counts)

fig, ax = plt.subplots(figsize=(5, 3))
fraud_counts.rename({0: 'Legit', 1: 'Fraud'}).plot.bar(ax=ax, color=['steelblue', 'crimson'])
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('../data/raw/class_distribution.png')
plt.show()

## 2. Missing Value Rates (top 30 columns)

In [ ]:
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print('Columns with > 50% missing:')
print(missing[missing > 50].to_string())

fig, ax = plt.subplots(figsize=(10, 4))
missing.head(30).plot.bar(ax=ax)
ax.set_title('Missing Value Rate (%) — Top 30 Columns')
ax.set_ylabel('% Missing')
ax.axhline(50, color='red', linestyle='--', alpha=0.6, label='50% threshold')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Transaction Amount Distribution

In [ ]:
print('TransactionAmt stats:')
print(df['TransactionAmt'].describe())
print(f'\nTraining mean  : {df["TransactionAmt"].mean():.2f}')
print(f'Training std   : {df["TransactionAmt"].std():.2f}')
print('  --> update TRAIN_AMT_MEAN / TRAIN_AMT_STD in api/preprocessing.py')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['TransactionAmt'].clip(upper=1000).hist(bins=60, ax=axes[0], color='steelblue')
axes[0].set_title('TransactionAmt (clipped at 1000)')
axes[0].set_xlabel('Amount (USD)')

np.log1p(df['TransactionAmt']).hist(bins=60, ax=axes[1], color='darkorange')
axes[1].set_title('log1p(TransactionAmt)')
axes[1].set_xlabel('log(1 + Amount)')

plt.tight_layout()
plt.show()

## 4. Amount by Fraud Label

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label, grp in df.groupby('isFraud'):
    np.log1p(grp['TransactionAmt']).hist(
        bins=50, ax=ax, alpha=0.6,
        label='Fraud' if label else 'Legit'
    )
ax.set_title('log(1 + Amount) by Fraud Label')
ax.set_xlabel('log(1 + Amount)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap (numeric, key columns)

In [ ]:
key_cols = ['TransactionAmt', 'TransactionDT', 'card1', 'card2',
            'addr1', 'dist1', 'C1', 'C2', 'D1', 'D15', 'V258', 'V308', 'isFraud']
corr = df[key_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, annot_kws={'size': 7})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Key Takeaways

Write your observations here after running:
- Fraud rate: ~3.5% → heavily imbalanced → **need SMOTE**
- Several V-columns have >80% missing → drop or indicator-encode
- `TransactionAmt` is right-skewed → use **log1p** transform
- `C1`, `C2`, `D1` correlate with fraud — good signals